# 02 - Entrenamiento del Modelo

**Proyecto 04 — Clasificación y Extracción de Documentos con OCR + IA**

Pipeline: TF-IDF (max_features=5000, ngram_range=(1,2)) + LinearSVC (C=1.0)
Validación cruzada de 5 pliegues (5-fold cross-validation)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

CATEGORIES = ['factura', 'recibo', 'contrato', 'constancia',
              'carta_formal', 'identificacion', 'otro']

# Cargar datos
texts, labels = [], []
for cat in CATEGORIES:
    cat_dir = Path(f'../data/training/{cat}')
    for f in cat_dir.glob('*.txt'):
        try:
            texts.append(f.read_text(encoding='utf-8'))
            labels.append(cat)
        except:
            pass

print(f'Total documentos cargados: {len(texts)}')
from collections import Counter
print('Por categoría:', Counter(labels))

In [ ]:
# Vectorización TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts)
print(f'Matriz TF-IDF: {X.shape}')
print(f'Vocabulario: {len(vectorizer.vocabulary_)} términos')

In [ ]:
# Validación cruzada 5-fold (LinearSVC)
svc = LinearSVC(C=1.0, max_iter=2000, random_state=42)
scores = cross_val_score(svc, X, labels, cv=5, scoring='f1_macro')
print(f'F1-macro (5-fold CV): {scores.mean():.4f} ± {scores.std():.4f}')
print(f'Scores por fold: {[f"{s:.4f}" for s in scores]}')

In [ ]:
# Entrenamiento final con split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tr = vec.fit_transform(X_train)
X_te = vec.transform(X_test)

svc_final = LinearSVC(C=1.0, max_iter=2000, random_state=42)
clf = CalibratedClassifierCV(svc_final, cv=5)
clf.fit(X_tr, y_train)

y_pred = clf.predict(X_te)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy en test: {acc:.4f} ({acc*100:.1f}%)')

## Resultados de Entrenamiento

- Algoritmo: TF-IDF + LinearSVC (C=1.0) con CalibratedClassifierCV
- Validación: 5-fold cross-validation
- Split final: 80% train / 20% test